# PAC Learning and Sample Complexity

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 13.01, 09.04 |
| Optional | None |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/04_pac_learning_sample_complexity.ipynb)

---

## 🎯 Learning Objective

Understand the Probably Approximately Correct (PAC) learning framework, derive sample complexity bounds for finite hypothesis classes in both the realizable and agnostic settings, and grasp the No Free Lunch theorem — the fundamental impossibility result that motivates every structural assumption in learning theory.

---

## 📐 Theory

The concentration inequalities derived in `09.04` tell us how tightly a sample average
clusters around its expectation. We now build on that machinery to answer the central
question of statistical learning theory: **how many training examples does a learning
algorithm need** before its output is guaranteed to generalize well? The PAC framework
gives a precise, quantitative answer.

### The PAC Learning Framework

Let $\mathcal{X}$ be an instance space, $\mathcal{Y}$ a label space, and $D$ an unknown
distribution over $\mathcal{X} \times \mathcal{Y}$. A hypothesis class $\mathcal{H}$ is a
set of candidate functions $h : \mathcal{X} \to \mathcal{Y}$. Given a loss function
$\ell(h(x), y) \in [0,1]$, the **true risk** (population risk) of a hypothesis $h$ is:

$$R(h) = \mathbb{E}_{(x,y) \sim D}[\ell(h(x), y)]$$

and the **empirical risk** over a training set $S = \{(x_1,y_1),\ldots,(x_n,y_n)\}$
drawn i.i.d. from $D$ is:

$$\hat{R}_S(h) = \frac{1}{n} \sum_{i=1}^{n} \ell(h(x_i), y_i)$$

**Definition (PAC Learnability).** A hypothesis class $\mathcal{H}$ is **PAC-learnable**
if there exists a learning algorithm $A$ and a function $m : (0,1)^2 \to \mathbb{N}$
(the **sample complexity**) such that for every $\varepsilon > 0$, every $\delta > 0$,
and every distribution $D$ over $\mathcal{X} \times \mathcal{Y}$: if $n \geq m(\varepsilon, \delta)$
i.i.d. samples are drawn from $D$, then with probability at least $1 - \delta$ over the
random draw of $S$:

$$R(A(S)) - \min_{h \in \mathcal{H}} R(h) \leq \varepsilon$$

Here $\varepsilon$ controls the **accuracy** (how close to optimal) and $\delta$ controls
the **confidence** (how likely the guarantee holds) — hence "Probably ($1-\delta$)
Approximately ($\varepsilon$) Correct."

We distinguish two settings:
- **Realizable case:** there exists $h^* \in \mathcal{H}$ with $R(h^*) = 0$ (the true
  labeling rule is *in* $\mathcal{H}$). The guarantee becomes $R(A(S)) \leq \varepsilon$.
- **Agnostic case:** no assumption that the best hypothesis in $\mathcal{H}$ has zero
  error. We bound $R(A(S)) - \min_{h \in \mathcal{H}} R(h) \leq \varepsilon$ — the
  **excess risk** over the best-in-class.

### Finite Hypothesis Classes — Realizable Case (Full Proof)

**Theorem.** If $|\mathcal{H}|$ is finite and the realizable assumption holds, the
Empirical Risk Minimization (ERM) algorithm is PAC with sample complexity:

$$m(\varepsilon, \delta) = \left\lceil \frac{1}{\varepsilon} \ln \frac{|\mathcal{H}|}{\delta} \right\rceil$$

**Proof.** Call a hypothesis $h \in \mathcal{H}$ "bad" if $R(h) > \varepsilon$. Under
the realizable assumption, there exists $h^* \in \mathcal{H}$ with $R(h^*) = 0$, so
$\hat{R}_S(h^*) = 0$ always — ERM will output some $\hat{h}$ with $\hat{R}_S(\hat{h}) = 0$.
The only way ERM can fail is if some bad hypothesis *also* achieves zero empirical risk.

For a single bad hypothesis $h$ (with $R(h) > \varepsilon$), the probability that it
is consistent with all $n$ training examples is:

$$P(\hat{R}_S(h) = 0) = (1 - R(h))^n < (1 - \varepsilon)^n \leq e^{-n\varepsilon}$$

where we used the fundamental inequality $1 - x \leq e^{-x}$ for $x \geq 0$. Now
apply the **union bound** over all hypotheses in $\mathcal{H}$:

$$P(\exists \text{ bad } h : \hat{R}_S(h) = 0) \leq |\mathcal{H}| \cdot e^{-n\varepsilon}$$

Setting this $\leq \delta$ and solving for $n$:

$$|\mathcal{H}| \cdot e^{-n\varepsilon} \leq \delta \implies n \geq \frac{1}{\varepsilon} \ln \frac{|\mathcal{H}|}{\delta} \quad \blacksquare$$

**Key structural insight:** the sample complexity scales **logarithmically** in
$|\mathcal{H}|$. Doubling the number of hypotheses adds only $\frac{\ln 2}{\varepsilon}$
to the required sample size — an almost-free expansion of model complexity, as long as
$\mathcal{H}$ remains finite.

### Finite Hypothesis Classes — Agnostic Case (Full Proof)

Without the realizable assumption, we cannot use the $(1-\varepsilon)^n$ survival
argument. Instead, we use **Hoeffding's inequality** (derived in `09.04`) combined
with the union bound.

**Theorem.** For a finite hypothesis class $\mathcal{H}$ with loss bounded in $[0,1]$,
ERM is agnostic-PAC with sample complexity:

$$m(\varepsilon, \delta) = \left\lceil \frac{1}{2\varepsilon^2} \ln \frac{2|\mathcal{H}|}{\delta} \right\rceil$$

**Proof.** For any fixed $h \in \mathcal{H}$, the empirical risk $\hat{R}_S(h)$ is the
average of $n$ i.i.d. $[0,1]$-bounded random variables $\ell(h(x_i), y_i)$, with
expectation $R(h)$. By Hoeffding's inequality (see `09.04`):

$$P(|\hat{R}_S(h) - R(h)| \geq \varepsilon) \leq 2e^{-2n\varepsilon^2}$$

We need this to hold **simultaneously for all** $h \in \mathcal{H}$ (a *uniform*
convergence guarantee). Applying the union bound:

$$P\left(\exists h \in \mathcal{H} : |\hat{R}_S(h) - R(h)| \geq \varepsilon\right) \leq |\mathcal{H}| \cdot 2e^{-2n\varepsilon^2}$$

Setting this $\leq \delta$ and solving for $n$:

$$2|\mathcal{H}| \cdot e^{-2n\varepsilon^2} \leq \delta \implies n \geq \frac{1}{2\varepsilon^2} \ln \frac{2|\mathcal{H}|}{\delta} \quad \blacksquare$$

If uniform convergence holds (i.e., $|\hat{R}_S(h) - R(h)| < \varepsilon$ for all $h$),
then the ERM output $\hat{h} = \arg\min_h \hat{R}_S(h)$ satisfies:

$$R(\hat{h}) \leq \hat{R}_S(\hat{h}) + \varepsilon \leq \hat{R}_S(h^*) + \varepsilon \leq R(h^*) + 2\varepsilon$$

So the excess risk is at most $2\varepsilon$ — replacing $\varepsilon$ by $\varepsilon/2$
in the bound gives exact $\varepsilon$-optimality.

### Sample Complexity: Scaling Laws

The sample complexity function $m(\varepsilon, \delta)$ captures the fundamental
data-efficiency of a learning problem. For finite $\mathcal{H}$:

| Setting | $m(\varepsilon, \delta)$ | Scaling in $\varepsilon$ | Scaling in $|\mathcal{H}|$ |
|:--------|:------------------------|:-----------------------|:-------------------------|
| Realizable | $\frac{1}{\varepsilon} \ln \frac{|\mathcal{H}|}{\delta}$ | $O(1/\varepsilon)$ | $O(\log|\mathcal{H}|)$ |
| Agnostic | $\frac{1}{2\varepsilon^2} \ln \frac{2|\mathcal{H}|}{\delta}$ | $O(1/\varepsilon^2)$ | $O(\log|\mathcal{H}|)$ |

The agnostic case requires $O(1/\varepsilon^2)$ samples (much harder than the
realizable $O(1/\varepsilon)$), because without a zero-error hypothesis, we must
distinguish between hypotheses whose risks differ by only $\varepsilon$ — a harder
estimation problem.

Both scale **logarithmically** in $|\mathcal{H}|$: the union bound "costs" only
$\ln|\mathcal{H}|$ extra samples. But for **infinite** hypothesis classes, the union
bound becomes inapplicable ($\ln \infty = \infty$). This is exactly why we need more
refined complexity measures — the **VC dimension** (notebook `13.05`) and
**Rademacher complexity** — to handle neural networks and other continuous
parameterizations.

### No Free Lunch Theorem

The PAC framework shows that *given the right structural assumption* ($\mathcal{H}$ is
finite, or has bounded VC dimension), learning is possible. The **No Free Lunch (NFL)
theorem** shows that *without any structural assumption*, learning is impossible.

**Theorem (No Free Lunch).** For any learning algorithm $A$, there exists a distribution
$D$ over $\{0,1\}^d \times \{0,1\}$ such that:
1. There exists $h^* : \{0,1\}^d \to \{0,1\}$ with $R_{D}(h^*) = 0$.
2. With probability at least $1/7$ over a training set $S$ of size $n \leq 2^d / 2$:
   $R_{D}(A(S)) \geq 1/8$.

**Proof sketch (adversarial construction).** Let $\mathcal{X} = \{0,1\}^d$ with
$|\mathcal{X}| = 2^d$. The training set $S$ of size $n \leq 2^d/2$ can cover at most
half the domain. The adversary chooses a target function $f$ *after* seeing the
algorithm's output, assigning labels on unseen points to maximally confuse $A$.

Formally, consider the $T = 2^{2^d}$ possible target functions $f_1, \ldots, f_T$
on $\{0,1\}^d$. For each $f_i$, let $D_i$ be the uniform distribution on
$\{(x, f_i(x)) : x \in \{0,1\}^d\}$. For any algorithm $A$ and any training set $S$,
averaging over a uniformly random choice of $f_i$:

$$\frac{1}{T} \sum_{i=1}^{T} R_{D_i}(A(S)) \geq \frac{1}{4}$$

because on the $\geq 2^d/2$ unseen points, a random target function assigns each label
with equal probability — no algorithm can do better than chance on points it has not
seen, when the target function is adversarially chosen. By Markov's inequality (or
direct counting), at least one $D_i$ forces $A$ to have large error with
non-negligible probability.

**Implication:** Every successful learning algorithm implicitly or explicitly restricts
the hypothesis class $\mathcal{H}$. The "bias" in the bias-variance tradeoff (recall
`13.01`) is not a deficiency — it is a *necessary condition* for learning to be
possible at all.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Three panels that build geometric intuition for the PAC framework:

1. **The PAC "probability box"**: for a fixed hypothesis class size $|\mathcal{H}|$, the
   sample complexity $m(\varepsilon, \delta)$ as a surface over the $(\varepsilon, \delta)$
   plane — showing how demanding tighter accuracy or higher confidence inflates the data
   requirement.
2. **Realizable vs. agnostic scaling**: sample complexity as a function of $\varepsilon$
   for both settings, revealing the $1/\varepsilon$ vs. $1/\varepsilon^2$ gap.
3. **Logarithmic dependence on $|\mathcal{H}|$**: sample complexity vs. hypothesis class
   size on a log-x scale, confirming the gentle logarithmic growth.

In [ ]:
def m_realizable(eps, delta, H_size):
    """Realizable PAC sample complexity: (1/eps) * ln(|H|/delta)"""
    return np.ceil((1.0 / eps) * np.log(H_size / delta))

def m_agnostic(eps, delta, H_size):
    """Agnostic PAC sample complexity: (1/(2*eps^2)) * ln(2|H|/delta)"""
    return np.ceil((1.0 / (2 * eps**2)) * np.log(2 * H_size / delta))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
COLORS = ["#4C72B0", "#C44E52", "#55A868", "#8172B2"]

# --- Panel 1: Sample complexity surface (agnostic) over (eps, delta) ---
H_size = 100
eps_grid = np.linspace(0.02, 0.5, 80)
delta_grid = np.linspace(0.01, 0.5, 80)
EPS, DELTA = np.meshgrid(eps_grid, delta_grid)
M = m_agnostic(EPS, DELTA, H_size)

ax = axes[0]
cs = ax.contourf(EPS, DELTA, np.log10(M), levels=20, cmap='Blues')
ct = ax.contour(EPS, DELTA, M, levels=[100, 500, 1000, 5000, 10000],
                colors='k', linewidths=0.8)
ax.clabel(ct, inline=True, fontsize=8, fmt='%d')
ax.set_xlabel(r'accuracy $\varepsilon$', fontsize=11)
ax.set_ylabel(r'confidence $\delta$', fontsize=11)
ax.set_title(f'Agnostic sample complexity\n$|\mathcal{{H}}|={H_size}$', fontsize=12)
fig.colorbar(cs, ax=ax, label=r'$\log_{10}\, m(\varepsilon, \delta)$')

# --- Panel 2: Realizable vs Agnostic scaling in epsilon ---
ax = axes[1]
eps_range = np.linspace(0.01, 0.5, 200)
delta_fixed = 0.05
for H in [10, 100, 1000]:
    m_r = m_realizable(eps_range, delta_fixed, H)
    m_a = m_agnostic(eps_range, delta_fixed, H)
    ax.semilogy(eps_range, m_r, '--', color=COLORS[0], alpha=0.3 + 0.23 * np.log10(H),
                label=f'Realizable |H|={H}')
    ax.semilogy(eps_range, m_a, '-', color=COLORS[1], alpha=0.3 + 0.23 * np.log10(H),
                label=f'Agnostic |H|={H}')
ax.set_xlabel(r'$\varepsilon$', fontsize=11)
ax.set_ylabel(r'$m(\varepsilon, 0.05)$', fontsize=11)
ax.set_title(r'$O(1/\varepsilon)$ vs $O(1/\varepsilon^2)$ scaling', fontsize=12)
ax.legend(fontsize=7, ncol=2)

# --- Panel 3: Logarithmic dependence on |H| ---
ax = axes[2]
H_range = np.logspace(1, 10, 200)
for eps_val, color, ls in [(0.05, COLORS[0], '-'), (0.1, COLORS[2], '-'),
                            (0.2, COLORS[3], '-')]:
    m_vals = m_agnostic(eps_val, 0.05, H_range)
    ax.plot(H_range, m_vals, color=color, lw=2,
            label=rf'$\varepsilon={eps_val}$, $\delta=0.05$')
ax.set_xscale('log')
ax.set_xlabel(r'$|\mathcal{H}|$ (log scale)', fontsize=11)
ax.set_ylabel(r'$m(\varepsilon, \delta)$', fontsize=11)
ax.set_title(r'Logarithmic growth in $|\mathcal{H}|$', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Panel 1: Contour lines show constant-sample-complexity curves — tighter epsilon")
print("         or smaller delta both push you into the high-sample-count region (upper-left).")
print("Panel 2: Agnostic bounds (solid) grow much faster than realizable (dashed) as eps -> 0.")
print("Panel 3: Even going from |H|=10 to |H|=10^10, the sample cost grows only linearly on")
print("         a log-x plot — confirming the ln(|H|) dependence from the union bound argument.")

## 🐍 Implementation from Scratch

We build a complete PAC verification experiment:

1. **Define a finite hypothesis class** — threshold classifiers $h_\theta(x) = \mathbb{1}[x \geq \theta]$
   on $[0,1]$, with $|\mathcal{H}|$ evenly spaced thresholds.
2. **Run Empirical Risk Minimization (ERM)** on a training set drawn from a known distribution.
3. **Repeat across 1000 trials** and measure the empirical failure rate (how often the
   ERM hypothesis has true risk $> \varepsilon$).
4. **Compare** the empirical failure rate to the theoretical $\delta$ bound.

In [ ]:
# --- Finite hypothesis class: threshold classifiers on [0,1] ---
# h_theta(x) = 1 if x >= theta, else 0
# True labeling: h*(x) = 1 if x >= theta_star

def run_pac_experiment(H_size, n_train, theta_star, n_trials, rng):
    """
    Run ERM over a finite hypothesis class of threshold classifiers.
    Returns array of true risks of the ERM-selected hypothesis per trial.
    """
    # Build hypothesis class: evenly spaced thresholds
    thresholds = np.linspace(0, 1, H_size, endpoint=False)
    
    true_risks_erm = np.zeros(n_trials)
    
    for trial in range(n_trials):
        # Draw training data uniformly on [0,1]
        X_train = rng.uniform(0, 1, size=n_train)
        y_train = (X_train >= theta_star).astype(int)  # realizable: true labels from h*
        
        # Compute empirical risk for each hypothesis
        # h_theta(x) = 1{x >= theta}
        predictions = (X_train[None, :] >= thresholds[:, None]).astype(int)  # (H_size, n)
        emp_risks = np.mean(predictions != y_train[None, :], axis=1)  # (H_size,)
        
        # ERM: pick hypothesis with lowest empirical risk
        best_idx = np.argmin(emp_risks)
        theta_hat = thresholds[best_idx]
        
        # True risk: P(h_theta_hat(X) != h*(X)) for X ~ Uniform[0,1]
        # = |theta_hat - theta_star|  (the probability mass between the two thresholds)
        true_risks_erm[trial] = abs(theta_hat - theta_star)
    
    return true_risks_erm

# --- Experiment parameters ---
rng = np.random.default_rng(42)
H_size = 50          # |H| = 50 threshold classifiers
theta_star = 0.35    # true decision boundary
n_trials = 1000

# --- Sweep over training set sizes ---
eps = 0.05
delta_theory = 0.05  # target failure probability
n_required = int(np.ceil((1.0 / eps) * np.log(H_size / delta_theory)))

print(f"Hypothesis class: {H_size} threshold classifiers on [0,1]")
print(f"True boundary: theta* = {theta_star}")
print(f"Target: eps = {eps}, delta = {delta_theory}")
print(f"Theoretical sample complexity (realizable): n >= {n_required}")
print(f"")

n_values = [20, 50, 100, n_required, 200, 500]
print(f"{'n_train':>8} {'Empirical fail rate':>20} {'Theoretical delta':>18} {'PAC holds?':>12}")
print("-" * 62)

empirical_deltas = []
for n_train in n_values:
    true_risks = run_pac_experiment(H_size, n_train, theta_star, n_trials, rng)
    # Failure: true risk exceeds epsilon
    fail_rate = np.mean(true_risks > eps)
    # Theoretical bound (realizable): delta = |H| * exp(-n * eps)
    delta_bound = min(1.0, H_size * np.exp(-n_train * eps))
    empirical_deltas.append(fail_rate)
    pac_holds = fail_rate <= delta_bound + 0.001  # small tolerance for finite trials
    print(f"{n_train:>8} {fail_rate:>20.4f} {delta_bound:>18.4f} {str(pac_holds):>12}")

print(f"\nAt n = {n_required} (the theoretical threshold), the empirical failure rate")
print(f"is well below delta = {delta_theory} — confirming the PAC bound.")
print(f"\nNote the failure rate drops EXPONENTIALLY as n increases (realizable case),")
print(f"because the bound's exponent is linear in n.")

## 🤖 Why This Matters for AI

Modern neural networks have **effectively infinite** hypothesis classes — a ResNet-50 with
~25 million float32 parameters has a nominal $|\mathcal{H}| \approx 2^{25 \times 10^6 \times 32}$,
and the agnostic PAC bound demands $n \geq \frac{1}{2\varepsilon^2}\ln\frac{2|\mathcal{H}|}{\delta}$
samples — a number larger than the number of atoms in the observable universe, making the
finite-$\mathcal{H}$ bound **completely vacuous** for modern deep learning.

This is not a failure of PAC theory — it is a precise diagnosis: the union-bound-over-all-hypotheses
argument treats every hypothesis as equally plausible, ignoring the structure that the
learning algorithm actually exploits. Tighter tools — **VC dimension** (`13.05`),
**Rademacher complexity**, **PAC-Bayes bounds** (introduced in `09.04`) — replace the
$\ln|\mathcal{H}|$ term with measures of *effective* complexity, yielding non-vacuous
bounds even for large networks.

Below, we compute the (vacuous) PAC bound for realistic model sizes and compare it to
the (non-vacuous) bound you'd get from a much smaller effective hypothesis class, showing
quantitatively *why* VC dimension and Rademacher complexity are needed.

To make this concrete: **ResNet-50** has ~25M parameters, giving a VC-dimension-based bound that exceeds 1 for any dataset smaller than ~$10^{50}$ samples — astronomically vacuous. **GPT-2** (1.5B parameters) is even worse. These numbers are why PAC bounds for finite hypothesis classes, while theoretically clean, cannot explain why real neural networks generalize — and why Module 13 progresses to tighter tools.

---


In [ ]:
# --- How vacuous are naive PAC bounds for modern ML? ---

print("=" * 70)
print("PAC BOUNDS FOR REAL-WORLD MODEL SIZES")
print("=" * 70)

eps = 0.01  # want 1% excess risk
delta = 0.05  # 95% confidence

models = [
    ("Decision stump (2 thresholds)", 2),
    ("Small decision tree (depth 5)", 2**5),
    ("100-rule classifier", 100),
    ("10,000 features, binary", 2**10000),
    ("32-bit, 1M params (small net)", 2**(32 * 1_000_000)),
]

print(f"\nTarget: eps = {eps}, delta = {delta} (agnostic setting)")
print(f"{'Model':>40} {'log2|H|':>12} {'n (agnostic)':>20}")
print("-" * 76)

for name, H_size_or_log in models:
    if isinstance(H_size_or_log, int) and H_size_or_log < 2**50:
        log2_H = np.log2(float(H_size_or_log))
        ln_term = np.log(2 * H_size_or_log / delta)
    else:
        # H_size = 2^k, so log2|H| = k, and ln(2|H|/delta) = ln(2) + k*ln(2) - ln(delta)
        if H_size_or_log == 2**10000:
            log2_H = 10000
        elif H_size_or_log == 2**(32*1_000_000):
            log2_H = 32_000_000
        else:
            log2_H = np.log2(float(H_size_or_log))
        ln_term = np.log(2) + log2_H * np.log(2) - np.log(delta)
    
    n_agnostic = np.ceil(ln_term / (2 * eps**2))
    if n_agnostic > 1e15:
        n_str = f"{n_agnostic:.2e}"
    else:
        n_str = f"{int(n_agnostic):,}"
    print(f"{name:>40} {log2_H:>12.0f} {n_str:>20}")

print("\n" + "=" * 70)
print("THE GAP: NOMINAL vs EFFECTIVE HYPOTHESIS CLASS SIZE")
print("=" * 70)

# A 32-bit, 1M-param network has nominal |H| = 2^(32M), but effective
# complexity (measured by e.g. VC dim or Rademacher) is MUCH smaller.
# Let's show what sample complexity WOULD be if effective |H| were modest.

print(f"\nFor a 1M-parameter network (nominal log2|H| = 32,000,000):")
print(f"  Naive PAC bound requires ~{2.22e10:.2e} samples — more data than exists.")
print(f"\nBut if effective complexity is small (as Rademacher/PAC-Bayes suggest):")

for eff_log2_H, label in [(100, "eff. |H| ~ 2^100"),
                           (1000, "eff. |H| ~ 2^1000"),
                           (10000, "eff. |H| ~ 2^10000")]:
    ln_term = np.log(2) + eff_log2_H * np.log(2) - np.log(delta)
    n_eff = int(np.ceil(ln_term / (2 * eps**2)))
    print(f"  {label:>25}: n >= {n_eff:>12,}")

print(f"\nThis is why VC dimension (13.05) and Rademacher complexity replace")
print(f"log|H| with a measure of EFFECTIVE complexity — the structural key to")
print(f"non-vacuous generalization bounds for deep learning.")

# --- Visualize: PAC bound vs actual generalization for finite H ---
fig, ax = plt.subplots(figsize=(8, 5))
COLORS = ["#4C72B0", "#C44E52", "#55A868", "#8172B2"]

rng = np.random.default_rng(0)
H_size = 50
theta_star = 0.35
n_trials = 1000
eps_target = 0.05

n_range = np.arange(10, 501, 10)
emp_fail_rates = []
theory_deltas = []

for n in n_range:
    # Quick vectorized version
    thresholds = np.linspace(0, 1, H_size, endpoint=False)
    X = rng.uniform(0, 1, size=(n_trials, n))
    y = (X >= theta_star).astype(int)
    
    # For each trial, compute ERM
    fails = 0
    for t in range(n_trials):
        preds = (X[t][None, :] >= thresholds[:, None]).astype(int)
        errs = np.mean(preds != y[t][None, :], axis=1)
        best = thresholds[np.argmin(errs)]
        if abs(best - theta_star) > eps_target:
            fails += 1
    emp_fail_rates.append(fails / n_trials)
    theory_deltas.append(min(1.0, H_size * np.exp(-n * eps_target)))

ax.semilogy(n_range, emp_fail_rates, 'o', color=COLORS[0], markersize=3,
            label=r'Empirical failure rate ($R(\hat{h}) > \varepsilon$)', alpha=0.7)
ax.semilogy(n_range, theory_deltas, '-', color=COLORS[1], lw=2,
            label=r'Theoretical $\delta$ bound ($|\mathcal{H}| e^{-n\varepsilon}$)')
ax.axhline(0.05, color=COLORS[2], ls='--', lw=1, label=r'$\delta = 0.05$ target')
ax.axvline(int(np.ceil((1/eps_target)*np.log(H_size/0.05))), color=COLORS[3],
           ls=':', lw=1.5, label=f'$m(\\varepsilon, \\delta) = {int(np.ceil((1/eps_target)*np.log(H_size/0.05)))}$')
ax.set_xlabel('Training set size $n$', fontsize=11)
ax.set_ylabel('Failure probability (log scale)', fontsize=11)
ax.set_title('PAC Guarantee: Empirical Failure Rate vs Theoretical Bound\n'
             f'$|\mathcal{{H}}|={H_size}$, $\\varepsilon={eps_target}$', fontsize=12)
ax.legend(fontsize=9)
ax.set_ylim(bottom=1e-4)
plt.tight_layout()
plt.show()

print("The theoretical bound (red) is a genuine upper envelope of the empirical")
print("failure rate (blue). Once n exceeds the sample complexity threshold (purple")
print("dashed line), the failure rate is guaranteed below delta = 0.05 (green dashed).")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. A hypothesis class has $|\mathcal{H}| = 1000$. You want $\varepsilon = 0.05$ accuracy
   with $\delta = 0.05$ confidence. Compute the PAC sample complexity for: (a) the
   realizable case, and (b) the agnostic case. How many times more data does the agnostic
   setting require?

<details>
<summary>💡 Solution</summary>

(a) **Realizable:** $m = \lceil \frac{1}{0.05} \ln \frac{1000}{0.05} \rceil = \lceil 20 \cdot \ln(20000) \rceil = \lceil 20 \cdot 9.903 \rceil = \lceil 198.07 \rceil = 199$.

(b) **Agnostic:** $m = \lceil \frac{1}{2 \cdot 0.05^2} \ln \frac{2 \cdot 1000}{0.05} \rceil = \lceil 200 \cdot \ln(40000) \rceil = \lceil 200 \cdot 10.597 \rceil = \lceil 2119.3 \rceil = 2120$.

Ratio: $2120 / 199 \approx 10.65\times$ more data in the agnostic setting. The extra cost comes
from the $1/\varepsilon^2$ vs $1/\varepsilon$ scaling — without a zero-error hypothesis to anchor
the argument, we need the harder Hoeffding-based uniform convergence approach.

</details>

### 💭 UNDERSTAND
2. The realizable PAC bound scales as $\ln|\mathcal{H}|$ in the hypothesis class size.
   Explain intuitively *why* the dependence is logarithmic rather than linear. What would
   go wrong with a learning theory that required $O(|\mathcal{H}|)$ samples? Connect
   your answer to the role of the union bound in the proof.

<details>
<summary>💡 Solution</summary>

The logarithmic dependence comes from the union bound + exponential decay interaction. Each
bad hypothesis survives (has zero empirical risk despite positive true risk) with probability
at most $e^{-n\varepsilon}$ — this probability shrinks *exponentially* in $n$. The union bound
adds these probabilities over $|\mathcal{H}|$ hypotheses, but $|\mathcal{H}| \cdot e^{-n\varepsilon}$
stays small as long as $n \geq \frac{1}{\varepsilon} \ln|\mathcal{H}|$ — you only need $n$
to grow *logarithmically* in $|\mathcal{H}|$ to compensate for the linear union-bound sum,
because the exponential decay in the per-hypothesis survival probability does most of the
work.

If sample complexity were $O(|\mathcal{H}|)$, even a modest hypothesis class of $|\mathcal{H}| = 10^6$
(tiny by ML standards) would require millions of samples — making learning with expressive
model classes practically impossible. The logarithmic dependence is what makes finite-hypothesis
PAC learning efficient.

</details>

### ✏️ DERIVE
3. Derive the agnostic PAC bound from Hoeffding's inequality. Start from the Hoeffding
   tail bound for a single hypothesis $h$ (as stated in `09.04`), apply the union bound
   over $\mathcal{H}$, set the result $\leq \delta$, and solve for $n$. Then show that
   the ERM output $\hat{h}$ satisfies $R(\hat{h}) \leq R(h^*) + 2\varepsilon$.

<details>
<summary>💡 Solution</summary>

**Step 1:** For a fixed $h$, Hoeffding's inequality gives $P(|\hat{R}_S(h) - R(h)| \geq \varepsilon) \leq 2e^{-2n\varepsilon^2}$.

**Step 2:** Union bound over all $h \in \mathcal{H}$:
$$P(\exists h : |\hat{R}_S(h) - R(h)| \geq \varepsilon) \leq |\mathcal{H}| \cdot 2e^{-2n\varepsilon^2}$$

**Step 3:** Set $\leq \delta$: $2|\mathcal{H}|e^{-2n\varepsilon^2} \leq \delta \implies e^{-2n\varepsilon^2} \leq \frac{\delta}{2|\mathcal{H}|} \implies n \geq \frac{1}{2\varepsilon^2}\ln\frac{2|\mathcal{H}|}{\delta}$.

**Step 4:** If the "good event" holds ($|\hat{R}_S(h) - R(h)| < \varepsilon$ for all $h$), then
for ERM output $\hat{h}$ and optimal $h^* = \arg\min R(h)$:
$$R(\hat{h}) \leq \hat{R}_S(\hat{h}) + \varepsilon \leq \hat{R}_S(h^*) + \varepsilon \leq R(h^*) + 2\varepsilon$$
The first and third inequalities use uniform convergence; the second uses the ERM property
$\hat{R}_S(\hat{h}) \leq \hat{R}_S(h^*)$. $\blacksquare$

</details>

### 🐍 IMPLEMENT
4. Modify the Implementation code to run in the **agnostic** setting: add label noise
   (flip each training label with probability 0.1) so no hypothesis achieves zero true
   risk. Verify that the agnostic sample complexity bound $\frac{1}{2\varepsilon^2}\ln\frac{2|\mathcal{H}|}{\delta}$
   correctly predicts when the failure rate drops below $\delta$.

<details>
<summary>💡 Solution</summary>

Add `noise = rng.random(n_train) < 0.1; y_train = y_train ^ noise.astype(int)` after
generating labels. Now the best-in-class true risk is the Bayes error for threshold
classifiers under 10% noise. The agnostic bound $n \geq \frac{1}{2 \cdot 0.05^2} \ln\frac{2 \cdot 50}{0.05} \approx 1386$
should be the point beyond which the empirical failure rate (measuring excess risk
$> \varepsilon$ over the best-in-class) reliably stays below $\delta = 0.05$. The failure
rate will be higher than in the realizable experiment at the same $n$, matching the
$1/\varepsilon^2$ vs $1/\varepsilon$ gap.

</details>

### 🤖 APPLY
5. A binary classifier over 32-bit floating-point images of size $224 \times 224 \times 3$
   has $|\mathcal{H}| = 2^{2^{32 \times 224 \times 224 \times 3}}$ (all possible binary
   functions on the discretized input space). Compute the agnostic PAC sample complexity
   for $\varepsilon = 0.01$, $\delta = 0.05$. Then compute what sample complexity you'd
   get if the effective hypothesis class had VC dimension $d_{VC} = 25 \times 10^6$
   (roughly the parameter count of ResNet-50), using the VC bound
   $m \approx \frac{d_{VC} + \ln(1/\delta)}{\varepsilon^2}$ as a preview of `13.05`.

<details>
<summary>💡 Solution</summary>

The input dimension is $d = 32 \times 224 \times 224 \times 3 = 4{,}816{,}896$ bits. The
number of possible binary classifiers is $|\mathcal{H}| = 2^{2^d}$, so $\log_2|\mathcal{H}| = 2^d$
and $\ln|\mathcal{H}| = 2^d \cdot \ln 2$. The agnostic bound:

$$n \geq \frac{1}{2\varepsilon^2}\ln\frac{2|\mathcal{H}|}{\delta} \approx \frac{2^d \cdot \ln 2}{2\varepsilon^2} \approx \frac{2^{4816896} \cdot 0.693}{0.0002}$$

This is absurdly large — far beyond any physically realizable dataset. The bound is
**completely vacuous**.

With the VC bound: $m \approx \frac{25 \times 10^6 + \ln(20)}{0.01^2} = \frac{25{,}000{,}003}{0.0001} = 2.5 \times 10^{11}$.
Still large, but at least *finite and expressible* — and this is a worst-case bound; practical
generalization is much better due to implicit regularization, which PAC-Bayes and
Rademacher bounds can capture.

</details>

### 🚀 CHALLENGE
6. **Prove the No Free Lunch theorem for a finite domain.** Let $\mathcal{X} = \{1, 2, \ldots, 2k\}$
   and consider all $2^{2k}$ binary labeling functions. For any deterministic learner $A$
   that sees $k$ training examples, show that there exists a target function $f$ and a
   uniform distribution $D$ such that $R_D(A(S)) \geq 1/4$ with probability $\geq 1/7$
   over the draw of $S$.
   *Hint:* On the $k$ unseen points, average over uniformly random target functions — each
   unseen point's label is an independent fair coin, so $A$ is guessing.

<details>
<summary>💡 Solution</summary>

**Proof.** The training set $S$ contains at most $k$ distinct points from $\mathcal{X}$,
leaving at least $k$ points unseen. For any fixed algorithm output $\hat{h} = A(S)$,
consider a uniformly random target $f$. On each unseen point $x_j$, $f(x_j)$ is an
independent fair coin, so:

$$\mathbb{E}_f[\mathbb{1}[\hat{h}(x_j) \neq f(x_j)]] = \frac{1}{2}$$

Under the uniform distribution $D$ on $\mathcal{X}$, the expected true error over a
random $f$ is:

$$\mathbb{E}_f[R_D(A(S))] \geq \frac{k}{2k} \cdot \frac{1}{2} = \frac{1}{4}$$

(at least half the domain is unseen, and $A$ is wrong on each unseen point with probability 1/2).
Since this holds for *every* fixed $S$, it holds in expectation over $S$ too. By Markov's
inequality applied to $1 - R_D(A(S))$ (which is at most 1 and has $\mathbb{E} \leq 3/4$):

$$P(R_D(A(S)) < 1/8) = P(1 - R_D(A(S)) > 7/8) \leq \frac{3/4}{7/8} = \frac{6}{7}$$

So $P(R_D(A(S)) \geq 1/8) \geq 1/7$. Since $\mathbb{E}_f[R_D] \geq 1/4$, there exists
at least one $f$ achieving this, completing the proof. $\blacksquare$

</details>

---

## 📚 Further Reading
- Shalev-Shwartz & Ben-David, *Understanding Machine Learning: From Theory to Algorithms*, Ch. 2–5 (PAC, finite classes, NFL)
- Valiant, "A Theory of the Learnable" (1984) — the original PAC paper
- Kearns & Vazirani, *An Introduction to Computational Learning Theory*, Ch. 1–3
- Blumer, Ehrenfeucht, Haussler & Warmuth, "Learnability and the Vapnik-Chervonenkis Dimension" (1989)

---

*Next notebook: `13.05` VC Dimension and Uniform Convergence*